# RSO-111: Analyze louver configurations

During the shutdown period (31.1.26 -14.2.26) we ran multiple versions (different louver configurations) of BLOCK-T679. This notebook creates a table and plot with said configurations.

**Description**

Output a table with the different configurations of louvers used and a simple plot showing the timeline.

**Expected results:**

Table: list of experiment segments with start/end time and louver configuration

Simple plot: louver configuration vs time (so we can visually confirm it makes sense)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from astropy.time import Time


from lsst.summit.utils.efdUtils import getEfdData, makeEfdClient
from lsst.summit.utils.tmaUtils import TMAEventMaker, TMAState

In [ ]:
t_start_period = Time("2026-01-31T00:00:00Z", scale="utc")
t_end_period = Time("2026-02-15T00:00:00Z", scale="utc")

efd_client = makeEfdClient()

# Queries

In [ ]:
def query_setlouvers(start, end):
    df_louvers = getEfdData(
        client=efd_client,
        topic="lsst.sal.MTDome.command_setLouvers",
        columns=["*"],
        begin=start,
        end=end,
    )

    return df_louvers

# Configuration of louvers

In [ ]:
df_setlouvers = query_setlouvers(t_start_period, t_end_period)

In [ ]:
# Copy current index into a new column before any merge
df_setlouvers['time_stamp'] = df_setlouvers.index

In [ ]:
# Select all columns that start with "position"
position_cols = df_setlouvers.filter(regex=r'^position').columns

# Sort columns
position_cols = sorted(position_cols, key=lambda x: int(x.replace('position', '')))

# Compute unique combinations
combination_counts = (
    df_setlouvers[position_cols]
    .value_counts()
    .reset_index(name='count')
)

# Create configuration ID (1 to N)
combination_counts['louvers_conf'] = range(1, len(combination_counts) + 1)

# Merge configuration ID back into original dataframe
df_setlouvers = df_setlouvers.merge(
    combination_counts[position_cols + ['louvers_conf']],
    on=position_cols,
    how='left'
)

print(f"Number of unique configurations detected: {len(combination_counts)}\n")

# Print configurations showing only non-zero positions
for _, row in combination_counts.iterrows():
    
    conf_id = row['louvers_conf']
    count = row['count']
    
    print(f"Configuration {conf_id} (appears {count} times):")
    
    # Extract position values
    config = row[position_cols]
    
    # Keep only non-zero values
    non_zero = config[config != 0]
    
    if len(non_zero) == 0:
        print("  All positions are 0")
    else:
        for col, val in non_zero.items():
            print(f"  {col}: {val}")
    
    print("-" * 40)

15 combinations have been made, varying the opening of the louvers: 2, 11, 12, 20, 21, and 29.

In [ ]:
df_temp0 = getEfdData(
        client=efd_client,
        topic="lsst.sal.ESS.temperature",
        columns=[f"temperatureItem0"],
        begin=t_start_period,
        end=t_end_period,
)


In [ ]:
fig, ax = plt.subplots(figsize=(12,6))
ax.plot(df_temp0.iloc[::100])
for i,t in enumerate(df_setlouvers['time_stamp']):
    if i==0:
        ax.axvline(t, color='red', alpha=0.3, label='Louver config set')
    else:
        ax.axvline(t, color='red', alpha=0.3)
ax.set_xlabel("Time")
ax.set_ylabel("Temperature sensor 0")
ax.set_title("Temperature with Louver Configurations")
ax.legend()
